# Spectral fitting example (AGN)


This notebook fits the spectrum of a simulated AGN observation with COSI using the same 3ML workflow as the Crab spectral fitting tutorial.

The binned mock data already include NGC 4151 plus background. The fit uses the mock histogram as the observed data and uses `mock - NGC4151` as the fitted background template, with NGC 4151 modeled as a cutoff-power-law thermal component plus an optional simple-power-law non-thermal tail.


In [43]:
from cosipy import BinnedData
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.event_selection import GoodTimeInterval
from cosipy.response.FullDetectorResponse import FullDetectorResponse
from cosipy.util import fetch_wasabi_file

from cosipy.statistics import PoissonLikelihood
from cosipy.background_estimation import FreeNormBinnedBackground
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.response import BinnedThreeMLModelFolding, BinnedInstrumentResponse, BinnedThreeMLPointSourceResponse
from cosipy.data_io import EmCDSBinnedData

from histpy import Histogram

import sys

import astropy.units as u
from astropy.coordinates import SkyCoord

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from threeML import PointSource, Model, JointLikelihood, DataList
from astromodels import Parameter, Cutoff_powerlaw, Powerlaw, Line

from pathlib import Path

%matplotlib inline


## Download and read in binned data


Define the paths to the mock data, original NGC 4151 data, detector response, orientation file, and yaml configuration. The mock data are expected to be binned before running this notebook.


In [148]:
analysis_dir = Path.cwd()
if not (analysis_dir / "agn.yaml").exists():
    repo_notebook_dir = Path("/Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/continuum_fit/AGN")
    if repo_notebook_dir.exists():
        analysis_dir = repo_notebook_dir

# binned_mock_data_file = Path("/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/Mock_Data/Mock_Data_Cut/mock_with_updated_NGC4151_FluxNTH_0p35_time_cut.hdf5")
# NGC4151_file = Path("/Users/parshadkp/Software/COSI_Data/DC4_Files/NGC_4151_ec200_0p35_DC3_COSI_cpl_pl_time_cut_in_fov.hdf5")

# binned_mock_data_file = Path("/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/Mock_Data/Mock_Data_Cut/mock_with_updated_NGC4151_FluxNTH_0p25_time_cut.hdf5")
# NGC4151_file = Path("/Users/parshadkp/Software/COSI_Data/DC4_Files/NGC_4151_ec200_0p25_DC4_COSI_cpl_pl_time_cut_in_fov.hdf5")

# binned_mock_data_file = Path("/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/Mock_Data/Mock_Data_Cut/mock_with_updated_NGC4151_FluxNTH_0p30_time_cut.hdf5")
# NGC4151_file = Path("/Users/parshadkp/Software/COSI_Data/DC4_Files/NGC_4151_ec200_0p30_DC4_COSI_cpl_pl_time_cut_in_fov.hdf5")

binned_mock_data_file = Path("/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/Mock_Data/Mock_Data_Cut/mock_with_updated_NGC4151_FluxNTH_0p15_time_cut.hdf5")
NGC4151_file = Path("/Users/parshadkp/Software/COSI_Data/DC4_Files/NGC_4151_ec200_DC4_COSI_cpl_pl_fluxNTH_0p15_time_cut_in_fov.hdf5")

# binned_mock_data_file = Path("/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/Mock_Data/Mock_Data_Cut/dc4_mock_dataset_3months_unbinned_data_filtered_with_SAAcut_time_ordered_ngc4151_in_fov_binned.hdf5")
# NGC4151_file = Path("/Users/parshadkp/Software/COSI_Data/DC4_Files/NGC_4151__3months_unbinned_data_filtered_with_SAAcut_ngc4151_in_fov_binned.hdf5")

orientation_path = "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.ori"
response_path = "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5"
agn_config = analysis_dir / "agn.yaml"

exposure_multiplier = 8
exposure_months = exposure_multiplier * 3

# Toggle whether the response uses the NGC 4151 FOV-cut spacecraft history.
use_time_cut_orientation = True
include_nonthermal_tail = True
tail_ratio_at_200keV = 0.15
max_offaxis = 60 * u.deg
earth_occ = True

source_label = "NGC 4151"
source_l = 155.07 * u.deg
source_b = 75.06 * u.deg
source_coord = SkyCoord(l=source_l, b=source_b, frame="galactic")

In [ ]:
total_mock_data = Path("/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/Mock_Data/Mock_Data_Cut/mock_with_updated_NGC4151_FluxNTH_0p15_time_cut.hdf5")
background_DC4 = Path("/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/Background/Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_withSAAbck_ngc4151_in_fov_binned.hdf5")
background_DC4_old = Path("/Users/parshadkp/Software/COSI_Data/DC4_Files/Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck_NGC4151_TimeCut.hdf5")
NGC_4151_old = Path("/Users/parshadkp/Software/COSI_Data/AGN_Data/GammaRay/Paper_Models/NGC_4151_ec200_DC3_COSI_cpl_pl.hdf5")
NGC_4151_DC4 = Path("/Users/parshadkp/Software/COSI_Data/DC4_Files/NGC_4151__3months_binned_data_filtered_with_SAAcut.hdf5")
full_mock_data = Path("/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/Mock_Data/dc4_mock_dataset_3months_binned_data_filtered_with_SAAcut_time_ordered.hdf5")

FONT_SIZE = 30
plt.rcParams["agg.path.chunksize"] = 10000
plt.rcParams.update({"font.size": FONT_SIZE})
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams.update({
    "font.weight": "550",
    "axes.titleweight": "550",
    "axes.labelweight": "550",
})

total_mock_data = Histogram.open(total_mock_data)
background_DC4 = Histogram.open(background_DC4)
background_DC4_old = Histogram.open(background_DC4_old)
NGC_4151_hist_comp = Histogram.open(NGC4151_file)
NGC_4151_hist_old = Histogram.open(NGC_4151_old)
NGC_4151_hist_DC4 = Histogram.open(NGC_4151_DC4)
full_mock_data_hist = Histogram.open(full_mock_data)
NGC_4151_hist_comp_2 = Histogram.open(NGC4151_file_2)
total_mock_data_2 = Histogram.open(binned_mock_data_file_2)


def projection_counts(hist, axis_name):
    return hist.project(axis_name).to_dense(copy=False).contents


def style_axis(ax):
    ax.xaxis.set_tick_params(which="major", size=12, width=1.5, direction="in", top=True, pad=8, labelsize=FONT_SIZE)
    ax.xaxis.set_tick_params(which="minor", size=6, width=1.5, direction="in", top=True, pad=8)
    ax.yaxis.set_tick_params(which="major", size=12, width=1.5, direction="in", right=True, pad=8, labelsize=FONT_SIZE)
    ax.yaxis.set_tick_params(which="minor", size=6, width=1.5, direction="in", right=True, pad=8)
    ax.spines["right"].set_visible(True)
    ax.spines["top"].set_visible(True)

mock_data_counts = projection_counts(total_mock_data, "Em")
mock_data_counts_2 = projection_counts(total_mock_data_2, "Em")
background_DC4_counts = projection_counts(background_DC4, "Em")
background_DC4_old_counts = projection_counts(background_DC4_old, "Em")
source_counts = projection_counts(NGC_4151_hist_comp, "Em")
source_counts_2 = projection_counts(NGC_4151_hist_comp_2, "Em")
source_old_counts = projection_counts(NGC_4151_hist_old, "Em")
source_DC4_counts = projection_counts(NGC_4151_hist_DC4, "Em")
full_mock_data_counts = projection_counts(full_mock_data_hist, "Em")

total_source_counts = source_counts.sum() * exposure_multiplier
total_source_old_counts = source_old_counts.sum() * exposure_multiplier
total_mock_data_counts = mock_data_counts.sum() * exposure_multiplier
total_background_counts = background_DC4_old_counts.sum() * exposure_multiplier
sigma = total_source_counts / np.sqrt(total_mock_data_counts)
sigma_old = total_source_old_counts / np.sqrt(total_source_old_counts + total_background_counts)
sigma_full = total_source_counts / np.sqrt(full_mock_data_counts.sum() * exposure_multiplier)

# print(f"Source counts: {total_source_counts:,.3g}")
# print(f"Background counts: {total_mock_data_counts:,.3g}")
print(f"Sigma (Mock Data): {sigma:.3f}")
print(f"Sigma (Single Source): {sigma_old:.3f}")
print(f"Sigma (Full Mock Data): {sigma_full:.3f}")

mock_data_edges = total_mock_data.axes["Em"].edges.to_value(u.keV)
background_DC4_edges = background_DC4.axes["Em"].edges.to_value(u.keV)

fig, ax = plt.subplots(figsize=(12, 9), constrained_layout=True)
style_axis(ax)
# ax.stairs(background_DC4_counts, background_DC4_edges / 1000, label="DC4 Background (Time Cut)", color="black", linestyle="dashdot")
ax.stairs(full_mock_data_counts, mock_data_edges / 1000, label="DC4 Mock Data (Full)", color="black", linestyle="solid")
ax.stairs(mock_data_counts, mock_data_edges / 1000, label="DC4 Mock Data (Time Cut)", color="red", linestyle="dashed")
ax.stairs(source_old_counts, mock_data_edges / 1000, label="NGC 4151 (Full)", color="green", linestyle="solid")
ax.stairs(source_counts, mock_data_edges / 1000, label="NGC 4151 (Time Cut)", color="blue", linestyle="dashed")
# ax.stairs(source_DC4_counts, mock_data_edges / 1000, label="NGC 4151 (DC4)", color="blue", linestyle="dashed")
# ax.stairs(source_counts_2, mock_data_edges / 1000, label="NGC 4151 (Time Cut, FluxNTH=0.35)", color="orange", linestyle="dashed")
# ax.stairs(source_counts, mock_data_edges / 1000, label="NGC 4151 (Time Cut, FluxNTH=0.15)", color="blue", linestyle="dashed")
# ax.stairs(mock_data_counts, mock_data_edges / 1000, label="DC4 Mock Data (Time Cut, FluxNTH=0.15)", color="red", linestyle="dashed")
# ax.stairs(mock_data_counts_2, mock_data_edges / 1000, label="DC4 Mock Data (Time Cut, FluxNTH=0.35)", color="black", linestyle="dashed")
ax.set_yscale("log")
ax.set_xscale("log")
ax.set_ylabel("Counts", fontsize=FONT_SIZE)
ax.set_ylim(1e6, 1e8)
ax.xaxis.set_major_locator(mticker.FixedLocator([0.2, 1.0, 5.0]))
ax.xaxis.set_major_formatter(mticker.FixedFormatter(["0.2", "1.0", "5.0"]))
ax.set_xlabel("Energy (MeV)", fontsize=FONT_SIZE)
ax.legend(fontsize=20, loc=(0.07, 0.02), frameon=False)

Exception ignored in: <function WeakValueDictionary.__init__.<locals>.remove at 0x11ec8ad40>
Traceback (most recent call last):
  File "/Users/parshadkp/.pyenv/versions/3.12.8/lib/python3.12/weakref.py", line 105, in remove
    def remove(wr, selfref=ref(self), _atomic_removal=_remove_dead_weakref):

KeyboardInterrupt: 


NameError: name 'NGC4151_file_2' is not defined

Read in the spacecraft orientation file.


In [149]:
# fetch_wasabi_file(
#     "COSI-SMEX/DC2/Data/Orientation/20280301_3_month_with_orbital_info.ori",
#     output=str(orientation_path),
#     checksum="416fcc296fc37a056a069378a2d30cb2",
# )
sc_orientation_full = SpacecraftHistory.open(orientation_path)
full_livetime = sc_orientation_full.cumulative_livetime().to_value(u.s)

if use_time_cut_orientation:
    source_gti = GoodTimeInterval.from_pointing_cut(
        source_coord,
        sc_orientation_full,
        max_offaxis,
        earth_occ=earth_occ,
    )
    sc_orientation = sc_orientation_full.apply_gti(source_gti)
    orientation_label = f"{source_label} FOV-cut .ori"
    orientation_color = "red"
else:
    sc_orientation = sc_orientation_full
    orientation_label = "Full .ori"
    orientation_color = "blue"

selected_livetime = sc_orientation.cumulative_livetime().to_value(u.s)

print(f"Full orientation livetime: {full_livetime:,.1f} s")
print(f"Selected orientation: {orientation_label}")
print(f"Selected orientation livetime: {selected_livetime:,.1f} s")
print(f"Selected/full livetime fraction: {selected_livetime / full_livetime:.4f}")


Full orientation livetime: 6,579,555.0 s
Selected orientation: NGC 4151 FOV-cut .ori
Selected orientation livetime: 980,415.0 s
Selected/full livetime fraction: 0.1490


Load the binned mock data and original NGC 4151 histogram. The mock histogram is the observed data, and `mock - NGC4151` is used as the fitted background template.


In [150]:
if not binned_mock_data_file.exists():
    raise FileNotFoundError(
        f"Set binned_mock_data_file to your binned mock data HDF5 file: {binned_mock_data_file}"
    )
if not NGC4151_file.exists():
    raise FileNotFoundError(
        f"Set NGC4151_file to your original binned NGC 4151 HDF5 file: {NGC4151_file}"
    )

mock = Histogram.open(binned_mock_data_file)
NGC4151 = Histogram.open(NGC4151_file)

data_hist = mock.project("Em", "Phi", "PsiChi") * exposure_multiplier
ngc4151_hist = NGC4151.project("Em", "Phi", "PsiChi") * exposure_multiplier
total_bkg = data_hist - ngc4151_hist


def make_background_distribution():
    # Build a fresh copy for each fit because FreeNormBinnedBackground normalizes its input.
    total_bkg_for_fit = total_bkg.copy()
    total_bkg_for_fit += sys.float_info.min
    return {"total_bkg": total_bkg_for_fit}


Fetch and open the response.


In [151]:
# fetch_wasabi_file(
#     "COSI-SMEX/develop/Data/Responses/ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5",
#     output=str(response_path),
#     checksum="7121f094be50e7bfe9b31e53015b0e85",
# )
dr = FullDetectorResponse.open(str(response_path))


## Perform spectral fit


Define a helper that sets the background parameter and instantiates the COSI 3ML plugin for the selected spacecraft history.


In [152]:
def build_cosi_plugin(sc_orientation, plugin_name="cosi"):
    # Wrap the raw BinnedData object into the appropriate data interface.
    data = EmCDSBinnedData(data_hist)

    # Use the background model to initialize a background expectation interface.
    bkg = FreeNormBinnedBackground(make_background_distribution(),
                                   sc_history=sc_orientation,
                                   copy=False)

    # Wrap the raw response with the BinnedInstrumentResponse interface implementation.
    instrument_response = BinnedInstrumentResponse(dr, data)

    # Initialize the 3ML point-source response.
    psr = BinnedThreeMLPointSourceResponse(data=data,
                                           instrument_response=instrument_response,
                                           sc_history=sc_orientation,
                                           energy_axis=dr.axes["Ei"],
                                           polarization_axis=dr.axes["Pol"] if "Pol" in dr.axes.labels else None,
                                           nside=2 * data.axes["PsiChi"].nside)

    # Pass the 3ML point-source response to the interface implementation that performs the folding with the spectrum.
    response = BinnedThreeMLModelFolding(data=data, point_source_response=psr)

    # Likelihood to use.
    like_fun = PoissonLikelihood(data, response, bkg)

    # Initialize 3ML plugin.
    cosi = ThreeMLPluginInterface(plugin_name,
                                  like_fun,
                                  response,
                                  bkg)

    # Initialize background parameters, considered as nuisance parameters.
    for bkg_label in bkg.labels:
        cosi.bkg_parameter[bkg_label] = Parameter(bkg_label,
                                                  200.0,
                                                  min_value=0.0,
                                                  max_value=10000.0,
                                                  delta=0.05,
                                                  unit=u.Hz)

    return {
        "data": data,
        "bkg": bkg,
        "instrument_response": instrument_response,
        "psr": psr,
        "response": response,
        "cosi": cosi,
    }


Define NGC 4151 at the known location with a cutoff-power-law thermal component and an optional power-law tail in a single composite point-source model.


Build the plugin and model for the CPL+PL and CPL-only hypotheses, combine each with a JointLikelihood object, and perform the maximum likelihood fits.


In [153]:
K_inj = (0.15 * exposure_multiplier) / u.cm / u.cm / u.s / u.keV
piv_inj = 1. * u.keV
xc_inj = 200. * u.keV
index_inj = -1.75

spectrum_inj_ec200 = Cutoff_powerlaw()

spectrum_inj_ec200.K.value = K_inj.value
spectrum_inj_ec200.piv.value = piv_inj.value
spectrum_inj_ec200.xc.value = xc_inj.value
spectrum_inj_ec200.index.value = index_inj

spectrum_inj_ec200.K.unit = K_inj.unit
spectrum_inj_ec200.piv.unit = piv_inj.unit
spectrum_inj_ec200.xc.unit = xc_inj.unit

K_inj = tail_ratio_at_200keV * spectrum_inj_ec200.evaluate_at(200) / u.cm / u.cm / u.s / u.keV
piv_inj = 200. * u.keV
index_inj = -3.8

spectrum_inj_ec200_PL = Powerlaw()

spectrum_inj_ec200_PL.K.value = K_inj.value
spectrum_inj_ec200_PL.piv.value = piv_inj.value
spectrum_inj_ec200_PL.index.value = index_inj

spectrum_inj_ec200_PL.K.unit = K_inj.unit
spectrum_inj_ec200_PL.piv.unit = piv_inj.unit

spectrum_inj_ec200_total = spectrum_inj_ec200 + spectrum_inj_ec200_PL

In [154]:
def build_ngc4151_model(include_tail=None):
    if include_tail is None:
        include_tail = include_nonthermal_tail

    # Give it some harsher initial guesses.
    index = -1.75
    reference_piv = 1. * u.keV
    piv = 200. * u.keV
    K = (
        (0.15 * exposure_multiplier)
        * (piv / reference_piv).to_value(u.dimensionless_unscaled) ** index
    ) / u.cm / u.cm / u.s / u.keV
    xc = 200. * u.keV

    spectrum_cpl = Cutoff_powerlaw()

    spectrum_cpl.K.value = K.value
    spectrum_cpl.piv.value = piv.value
    spectrum_cpl.xc.value = xc.value
    spectrum_cpl.index.value = index
    spectrum_cpl.index.fix = True

    # Keep the hard bounds broad enough that covariance samples are not clipped.
    spectrum_cpl.K.min_value = 1e-8
    spectrum_cpl.K.max_value = 1e-2
    spectrum_cpl.xc.min_value = 100
    spectrum_cpl.xc.max_value = 10000

    spectrum_cpl.K.unit = K.unit
    spectrum_cpl.piv.unit = piv.unit
    spectrum_cpl.xc.unit = xc.unit

    spectral_shape = spectrum_cpl
    tail_link_ratio = None

    if include_tail:
        # Put the tail in the same point source as the thermal component.
        # This matches the injected model structure and avoids fitting two linked point sources at the same sky position.
        K_tail = tail_ratio_at_200keV * spectrum_cpl.evaluate_at(200) / u.cm / u.cm / u.s / u.keV
        piv_tail = 200. * u.keV
        index_tail = -3.8

        spectrum_tail = Powerlaw()
        spectrum_tail.K.value = K_tail.value
        spectrum_tail.piv.value = piv_tail.value
        spectrum_tail.index.value = index_tail

        spectrum_tail.K.min_value = 1e-12
        spectrum_tail.K.max_value = 10
        spectrum_tail.index.min_value = -5
        spectrum_tail.index.max_value = -2
        spectrum_tail.index.delta = 0.25

        spectrum_tail.K.unit = K_tail.unit
        spectrum_tail.piv.unit = piv_tail.unit

        tail_link_ratio = K_tail.value / spectrum_cpl.K.value

        spectral_shape = spectrum_cpl + spectrum_tail

    ngc4151 = PointSource("ngc4151",
                          l=source_l.to_value(u.deg),
                          b=source_b.to_value(u.deg),
                          spectral_shape=spectral_shape)

    model = Model(ngc4151)

    if include_tail:
        link_function = Line(a=0.0, b=tail_link_ratio)
        link_function.a.fix = True
        link_function.b.min_value = 0.0
        link_function.b.max_value = 1.0
        link_function.b.delta = max(abs(tail_link_ratio) * 0.1, 1e-8)
        model.link(
            model.ngc4151.spectrum.main.composite.K_2,
            model.ngc4151.spectrum.main.composite.K_1,
            link_function,
        )

    return model


In [155]:
def fit_ngc4151_hypothesis(label, plugin_name, include_tail, color, linestyle):
    print(f"\nFitting {label} ...")

    fit = {
        "label": label,
        "sc_orientation": sc_orientation,
        "include_nonthermal_tail": include_tail,
        "color": color,
        "linestyle": linestyle,
    }
    fit.update(build_cosi_plugin(sc_orientation, plugin_name=plugin_name))
    fit["model"] = build_ngc4151_model(include_tail=include_tail)
    fit["plugins"] = DataList(fit["cosi"])
    fit["like"] = JointLikelihood(fit["model"], fit["plugins"], verbose=False)

    _ = fit["like"].fit()
    fit["results"] = fit["like"].results
    fit["expectation"] = fit["response"].expectation()

    return fit


cpl_pl_fit = fit_ngc4151_hypothesis(
    f"{orientation_label} CPL + PL",
    plugin_name="cosi_cpl_pl",
    include_tail=True,
    color=orientation_color,
    linestyle="dashdot",
)
cpl_only_fit = fit_ngc4151_hypothesis(
    f"{orientation_label} CPL only",
    plugin_name="cosi_cpl_only",
    include_tail=False,
    color="black",
    linestyle="dashed",
)

primary_fit = cpl_pl_fit

data = primary_fit["data"]
bkg = primary_fit["bkg"]
response = primary_fit["response"]
cosi = primary_fit["cosi"]
like = primary_fit["like"]
results = primary_fit["results"]
expectation = primary_fit["expectation"]


Fitting NGC 4151 FOV-cut .ori CPL + PL ...


01:04:32 INFO      set the minimizer to minuit                                             ]8;id=699644;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=89248;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

01:04:52 WARNING   20.560000000000002 percent of samples have been thrown away because     ]8;id=919786;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/analysis_results.py\analysis_results.py]8;;\:]8;id=204030;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/analysis_results.py#1645\1645]8;;\
                  they failed the constraints on the parameters. This results might not be                         
                  suitable for error propagation. Enlarge the boundaries until you loose                           
                  less than 1 percent of the samples.                                                              

01:04:52 WARNING   The current value of the parameter K_1 (1.0) was above the new maximum 0.01.    ]8;id=13596;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=145049;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py#810\810]8;;\

         WARNING   The current value of the parameter xc_1 (10.0) was below the new minimum 100.0. ]8;id=707478;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=760349;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py#732\732]8;;\

...

1.00,-0.91,-0.00
-0.91,1.00,-0.28
-0.00,-0.28,1.00


Values of -log(likelihood) at the minimum:

,-log(likelihood)
cosi_cpl_only,-1380855295.7805686
total,-1380855295.7805686


Values of statistical measures:

,statistical measures
AIC,-2761710585.5610332
BIC,-2761710554.5184207


## Non-thermal tail TS comparison


In [156]:
def get_likelihood_statistic(joint_likelihood):
    statistic_frame = joint_likelihood.results.get_statistic_frame()
    statistic_series = statistic_frame["-log(likelihood)"]

    if "total" in statistic_series.index:
        return float(statistic_series.loc["total"])

    return float(statistic_series.sum())


def model_improvement_ts(reference_likelihood, test_likelihood):
    return 2.0 * (
        get_likelihood_statistic(reference_likelihood)
        - get_likelihood_statistic(test_likelihood)
    )


stat_cpl = get_likelihood_statistic(cpl_only_fit["like"])
stat_cpl_pl = get_likelihood_statistic(cpl_pl_fit["like"])
TS_cpl_pl_vs_cpl = model_improvement_ts(cpl_only_fit["like"], cpl_pl_fit["like"])
tail_detection_sigma = np.sqrt(max(TS_cpl_pl_vs_cpl, 0.0))

tail_ts_comparison = pd.DataFrame(
    [
        {
            "spectrum": source_label,
            "fit": orientation_label,
            "minus_log_like_CPL": stat_cpl,
            "minus_log_like_CPL_plus_PL": stat_cpl_pl,
            "Delta_TS_CPL_plus_PL_vs_CPL": TS_cpl_pl_vs_cpl,
            "Sigma_added_PL": tail_detection_sigma,
        }
    ]
)

display(tail_ts_comparison)
print(
    "Added PL tail improvement TS: "
    f"{TS_cpl_pl_vs_cpl:.3f} ({tail_detection_sigma:.2f} sigma)"
)


,spectrum,fit,minus_log_like_CPL,minus_log_like_CPL_plus_PL,Delta_TS_CPL_plus_PL_vs_CPL,Sigma_added_PL
0,NGC 4151,NGC 4151 FOV-cut .ori,-1.380855e+09,-1.380855e+09,2.967968,1.722779


Added PL tail improvement TS: 2.968 (1.72 sigma)


## Error propagation and plotting


The summary of the results above gives the optimal values of the parameters, as well as the errors. Propagate the errors to the `evaluate_at` method of each fitted source spectrum.


In [157]:
results = primary_fit["results"]

print(f"\n{primary_fit['label']} fit results")
print(results.display())

source_parameters = {par.name: results.get_variates(par.path)
                     for par in results.optimized_model["ngc4151"].parameters.values()
                     if par.free}
primary_fit["flux_errors"] = results.propagate(
    results.optimized_model["ngc4151"].spectrum.main.shape.evaluate_at,
    **source_parameters,
)

print(results.optimized_model["ngc4151"])



NGC 4151 FOV-cut .ori CPL + PL fit results


Best fit values:

,result,unit
parameter,,
ngc4151.spectrum.main.composite.K_1,(1.13 -0.24 +0.30) x 10^-4,1 / (keV s cm2)
ngc4151.spectrum.main.composite.xc_1,(2.00 -0.24 +0.27) x 10^2,keV
ngc4151.spectrum.main.composite.K_2.Line.b,(6 +/- 6) x 10^-2,
ngc4151.spectrum.main.composite.index_2,-3.8 +/- 0.8,
total_bkg,(2.13047 +/- 0.00021) x 10^2,Hz


Correlation matrix:

1.00,-0.84,-0.88,-0.06,0.24
-0.84,1.00,0.50,-0.43,-0.31
-0.88,0.50,1.00,0.49,-0.23
-0.06,-0.43,0.49,1.00,-0.02
0.24,-0.31,-0.23,-0.02,1.00


...

         WARNING   The current value of the parameter K_1 (1.0) was above the new maximum 0.01.    ]8;id=344541;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=79218;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py#810\810]8;;\

         WARNING   The current value of the parameter xc_1 (10.0) was below the new minimum 100.0. ]8;id=114454;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=884692;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py#732\732]8;;\

         WARNING   The current value of the parameter K_1 (1.0) was above the new maximum 0.01.    ]8;id=951756;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=864454;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py#810\810]8;;\

         WARNING   The current value of the parameter xc_1 (10.0) was below the new minimum 100.0. ]8;id=307175;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=986712;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py#732\732]8;;\

  * ngc4151 (point source):
    * position:
      * l:
        * value: 155.07
        * desc: Galactic longitude
        * min_value: 0.0
        * max_value: 360.0
        * unit: deg
        * is_normalization: false
      * b:
        * value: 75.06
        * desc: Galactic latitude
        * min_value: -90.0
        * max_value: 90.0
        * unit: deg
        * is_normalization: false
      * equinox: J2000
    * spectrum:
      * main:
        * composite:
          * K_1:
            * value: 0.00011254995386407924
            * desc: Normalization (differential flux at the pivot value)
            * min_value: 1.0e-08
            * max_value: 0.01
            * unit: keV-1 s-1 cm-2
            * is_normalization: true
          * piv_1:
            * value: 200.0
            * desc: Pivot value
            * min_value: null
            * max_value: null
            * unit: keV
            * is_normalization: false
          * index_1:
            * value: -1.75
            * 

Evaluate the flux and errors at a range of energies for the fitted and injected spectra.


In [ ]:
energy = np.geomspace(200 * u.keV, 5 * u.MeV).to_value(u.keV)
flux_inj_thermal = np.zeros_like(energy)
flux_inj_tail = np.zeros_like(energy)
flux_inj = np.zeros_like(energy)

for i, e in enumerate(energy):
    flux_inj_thermal[i] = spectrum_inj_ec200.evaluate_at(e)

    if include_nonthermal_tail:
        flux_inj_tail[i] = spectrum_inj_ec200_PL.evaluate_at(e)
        flux_inj[i] = spectrum_inj_ec200_total.evaluate_at(e)
    else:
        flux_inj[i] = flux_inj_thermal[i]

flux_lo = np.zeros_like(energy)
flux_median = np.zeros_like(energy)
flux_hi = np.zeros_like(energy)

for i, e in enumerate(energy):
    flux = primary_fit["flux_errors"](e)
    flux_median[i] = flux.median
    flux_lo[i], flux_hi[i] = flux.equal_tail_interval(cl=0.68)

fit_thermal_flux = np.zeros_like(energy)
fit_tail_flux = np.zeros_like(energy)

optimized_shape = results.optimized_model["ngc4151"].spectrum.main.shape
if primary_fit["include_nonthermal_tail"]:
    fit_thermal_shape, fit_tail_shape = optimized_shape.functions

    for i, e in enumerate(energy):
        fit_thermal_flux[i] = fit_thermal_shape.evaluate_at(e)
        fit_tail_flux[i] = fit_tail_shape.evaluate_at(e)
else:
    for i, e in enumerate(energy):
        fit_thermal_flux[i] = optimized_shape.evaluate_at(e)

primary_fit["flux_median"] = flux_median
primary_fit["flux_lo"] = flux_lo
primary_fit["flux_hi"] = flux_hi
primary_fit["fit_thermal_flux"] = fit_thermal_flux
primary_fit["fit_tail_flux"] = fit_tail_flux
primary_fit["injected_thermal_flux"] = flux_inj_thermal
primary_fit["injected_tail_flux"] = flux_inj_tail

binned_energy_edges = data_hist.axes["Em"].edges.value
binned_energy = 0.5 * (binned_energy_edges[1:] + binned_energy_edges[:-1])

data = primary_fit["data"]
bkg = primary_fit["bkg"]
response = primary_fit["response"]
expectation = primary_fit["expectation"]


Plot the fitted and injected SEDs using the Paper_Plots component style. The injected curves show the thermal component, non-thermal tail, and total model; the fitted curves show the optimized total with its 68% interval and component best fits.


In [ ]:
FONT_SIZE = 30
plt.rcParams["agg.path.chunksize"] = 10000
plt.rcParams.update({"font.size": FONT_SIZE})
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams.update({
    "font.weight": "550",
    "axes.titleweight": "550",
    "axes.labelweight": "550",
})


def style_axis(ax):
    ax.xaxis.set_tick_params(which="major", size=12, width=1.5, direction="in", top=True, pad=8, labelsize=FONT_SIZE)
    ax.xaxis.set_tick_params(which="minor", size=6, width=1.5, direction="in", top=True, pad=8)
    ax.yaxis.set_tick_params(which="major", size=12, width=1.5, direction="in", right=True, pad=8, labelsize=FONT_SIZE)
    ax.yaxis.set_tick_params(which="minor", size=6, width=1.5, direction="in", right=True, pad=8)
    ax.spines["right"].set_visible(True)
    ax.spines["top"].set_visible(True)


fig, ax = plt.subplots(figsize=(12, 9), constrained_layout=True)
style_axis(ax)

ax.plot(
    energy / 1000,
    energy * energy * flux_inj_thermal,
    color="tab:red",
    ls=":",
    lw=3,
    label="Injected thermal",
)

if include_nonthermal_tail:
    ax.plot(
        energy / 1000,
        energy * energy * flux_inj_tail,
        color="tab:blue",
        ls="-.",
        lw=3,
        label="Injected non-thermal",
    )
    ax.plot(
        energy / 1000,
        energy * energy * flux_inj,
        color="black",
        ls="--",
        lw=3,
        label="Injected total",
    )

ax.plot(
    energy / 1000,
    energy * energy * primary_fit["flux_median"],
    color=primary_fit["color"],
    lw=2,
    linestyle=primary_fit["linestyle"],
    label="Best-fit total",
)
ax.fill_between(
    energy / 1000,
    energy * energy * primary_fit["flux_lo"],
    energy * energy * primary_fit["flux_hi"],
    alpha=0.18,
    color=primary_fit["color"],
    label="68% interval",
)

if primary_fit["include_nonthermal_tail"]:
    ax.plot(
        energy / 1000,
        energy * energy * primary_fit["fit_thermal_flux"],
        color="tab:red",
        lw=1.8,
        ls="--",
        alpha=0.75,
        label="Best-fit thermal",
    )
    ax.plot(
        energy / 1000,
        energy * energy * primary_fit["fit_tail_flux"],
        color="tab:blue",
        lw=1.8,
        ls="--",
        alpha=0.75,
        label="Best-fit non-thermal",
    )
ax.set_xscale("log")
ax.set_yscale("log")

ax.xaxis.set_major_locator(mticker.FixedLocator([0.2, 1.0, 5.0]))
ax.xaxis.set_major_formatter(mticker.FixedFormatter(["0.2", "1.0", "5.0"]))

ax.set_xlabel("Energy (MeV)", fontsize=FONT_SIZE)
ax.set_ylabel(r"Energy Flux (keV cm$^{-2}$ s$^{-1}$)", fontsize=FONT_SIZE)

ax.set_ylim(1e-4, 10)

def integrate_energy_flux(photon_flux):
    return np.trapezoid(energy * photon_flux, energy)

energy_flux_ratio = 0.0
if include_nonthermal_tail:
    thermal_flux = np.asarray([spectrum_inj_ec200.evaluate_at(e) for e in energy])
    tail_flux = np.asarray([spectrum_inj_ec200_PL.evaluate_at(e) for e in energy])
    thermal_energy_flux = integrate_energy_flux(thermal_flux)
    tail_energy_flux = integrate_energy_flux(tail_flux)
    energy_flux_ratio = tail_energy_flux / (tail_energy_flux + thermal_energy_flux)

ax.text(
    0.98,
    0.98,
    f"NGC 4151\n{exposure_months}-months\nNT Flux Fraction = {energy_flux_ratio:.2f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=FONT_SIZE,
    fontweight="550",
)

_ = ax.legend(fontsize=23, loc="lower left", frameon=False)

print(energy_flux_ratio)


Plot the fitted spectrum convolved with the response plus the fitted background, as well as the simulated source+background counts.
